# Spike de exploracion - `config_contract`

> **Paso 4 del flujo** (`notebook_writer`). Prototipo **visual** que demuestra end-to-end las historias `HU-01 ... HU-08` de `definition.md`, para el gate humano del paso 5.

## Que se prototipa
El **motor de validacion (Pydantic)** que lee un `contract_data.yaml` (Contrato de Datos, YAML 1 de 4) y valida que este **bien formado en si mismo**: lista de columnas, cada una con `nombre`, `tipo`, `nulabilidad` y `llave`. Devuelve un **objeto tipado en memoria** o un **error claro y accionable**.

## Regla inviolable - Datos en Boveda (C-01)
Este notebook usa **exclusivamente YAMLs sinteticos** (matrices de mentiras) escritos aqui mismo: contratos validos e invalidos con columnas ficticias (`test_id`, `Usuario 1`, `correo`...). **Ningun dato real de cliente entra a este notebook.** Nada se lee de `clients/<CLIENTE>/data/`.

## Naturaleza de spike (Opcion A)
Explora, no es producto. Al pasar a `app/src/` la spec y el bucle TDD **reescriben** la logica; este notebook queda como documentacion de referencia.

## Celda 1 - Motor: modelo Pydantic + funcion core `load_contract`

Traza -> **HU-07** (logica core reutilizable, invocable sin CLI) y base de todas las demas HU.

Define el enum de tipos soportados, los modelos `Columna` y `Contract`, dos excepciones que **distinguen parseo vs esquema** (HU-04) y la funcion `load_contract(path) -> Contract`.

In [1]:
from enum import Enum
from pathlib import Path
from typing import List

import yaml
from pydantic import BaseModel, ValidationError, field_validator


# --- Enum de tipos soportados (conjunto cerrado; la lista final se fija en la spec, HU-03) ---
class TipoDato(str, Enum):
    string = "string"
    integer = "integer"
    float = "float"
    date = "date"
    boolean = "boolean"


# --- Modelo de una columna del contrato ---
class Columna(BaseModel):
    nombre: str
    tipo: TipoDato
    nulable: bool
    llave: bool


# --- Modelo del contrato: lista de columnas con reglas estructurales ---
class Contract(BaseModel):
    columnas: List[Columna]

    @field_validator("columnas")
    @classmethod
    def lista_no_vacia(cls, v):
        if not v:
            raise ValueError("la lista de columnas no puede estar vacia")
        return v

    @field_validator("columnas")
    @classmethod
    def sin_nombres_duplicados(cls, v):
        nombres = [c.nombre for c in v]
        dups = sorted({n for n in nombres if nombres.count(n) > 1})
        if dups:
            raise ValueError(f"columnas duplicadas por nombre: {dups}")
        return v


# --- Excepciones que distinguen los dos modos de fallo (HU-04) ---
class ContractParseError(Exception):
    """YAML sintacticamente roto: falla antes de llegar al esquema."""


class ContractSchemaError(Exception):
    """YAML bien formado pero contrato mal formado: falla la validacion Pydantic."""


# --- Funcion core: unica puerta de entrada (HU-07) ---
def load_contract(path) -> Contract:
    """Lee y valida el contract_data.yaml de un tenant.

    Devuelve un Contract tipado si es valido; si no, lanza ContractParseError
    (sintaxis YAML) o ContractSchemaError (esquema del contrato).
    """
    texto = Path(path).read_text(encoding="utf-8")
    try:
        crudo = yaml.safe_load(texto)
    except yaml.YAMLError as e:
        raise ContractParseError(f"YAML sintacticamente invalido: {e}") from e
    cuerpo = (crudo or {}).get("contract_data") or {}
    try:
        return Contract.model_validate(cuerpo)
    except ValidationError as e:
        raise ContractSchemaError(str(e)) from e


print("Motor cargado: TipoDato =", [t.value for t in TipoDato])
print("Excepciones:", ContractParseError.__name__, "/", ContractSchemaError.__name__)

Motor cargado: TipoDato = ['string', 'integer', 'float', 'date', 'boolean']
Excepciones: ContractParseError / ContractSchemaError


## Celda 2 - Datos sinteticos (matrices de mentiras)

Traza -> **HU-08** (fixtures 100% sinteticos, sin PII, versionables).

Escribimos todos los fixtures YAML en un directorio **temporal** creado por el notebook. Ninguno proviene de un cliente real; ninguna ruta apunta a `clients/*/data/`.

In [2]:
import tempfile

FIXTURES = Path(tempfile.mkdtemp(prefix="config_contract_spike_"))

# 1) Contrato VALIDO (todas las columnas bien formadas) - datos ficticios
yaml_valido = """
contract_data:
  columnas:
    - nombre: test_id
      tipo: integer
      nulable: false
      llave: true
    - nombre: correo
      tipo: string
      nulable: false
      llave: false
    - nombre: monto
      tipo: float
      nulable: true
      llave: false
    - nombre: fecha_alta
      tipo: date
      nulable: true
      llave: false
    - nombre: activo
      tipo: boolean
      nulable: false
      llave: false
"""

# 2) Campo requerido faltante: la columna 'correo' no declara 'tipo'
yaml_campo_faltante = """
contract_data:
  columnas:
    - nombre: test_id
      tipo: integer
      nulable: false
      llave: true
    - nombre: correo
      nulable: false
      llave: false
"""

# 3) Tipo fuera del enum soportado: 'numero_magico'
yaml_tipo_invalido = """
contract_data:
  columnas:
    - nombre: test_id
      tipo: numero_magico
      nulable: false
      llave: true
"""

# 4) YAML sintacticamente roto (indentacion/estructura invalida)
yaml_roto = """
contract_data:
  columnas:
    - nombre: test_id
      tipo: integer
     nulable: false
       llave: true
"""

# 5a) Columnas duplicadas por nombre
yaml_duplicadas = """
contract_data:
  columnas:
    - nombre: test_id
      tipo: integer
      nulable: false
      llave: true
    - nombre: test_id
      tipo: string
      nulable: true
      llave: false
"""

# 5b) Lista de columnas vacia
yaml_vacio = """
contract_data:
  columnas: []
"""

fixtures = {
    "valido.yaml": yaml_valido,
    "campo_faltante.yaml": yaml_campo_faltante,
    "tipo_invalido.yaml": yaml_tipo_invalido,
    "roto.yaml": yaml_roto,
    "duplicadas.yaml": yaml_duplicadas,
    "vacio.yaml": yaml_vacio,
}
for nombre, contenido in fixtures.items():
    (FIXTURES / nombre).write_text(contenido, encoding="utf-8")

print(f"Fixtures sinteticos escritos en: {FIXTURES}")
for nombre in fixtures:
    print(" -", nombre)

Fixtures sinteticos escritos en: C:\Users\USUARIO\AppData\Local\Temp\config_contract_spike_e89tfijx
 - valido.yaml
 - campo_faltante.yaml
 - tipo_invalido.yaml
 - roto.yaml
 - duplicadas.yaml
 - vacio.yaml


## Celda 3 - HU-01: contrato valido -> objeto tipado en memoria

Traza -> **HU-01**. Cargamos el YAML valido y mostramos, en una **tabla**, que la lista de columnas del objeto tipado refleja fielmente lo declarado (nombre, tipo, nulabilidad, llave).

In [3]:
import pandas as pd

contrato = load_contract(FIXTURES / "valido.yaml")

print("Tipo del objeto devuelto:", type(contrato).__name__)
print("Numero de columnas:", len(contrato.columnas))

tabla = pd.DataFrame(
    [
        {
            "nombre": c.nombre,
            "tipo": c.tipo.value,
            "nulable": c.nulable,
            "llave": c.llave,
        }
        for c in contrato.columnas
    ]
)
tabla

Tipo del objeto devuelto: Contract
Numero de columnas: 5


,nombre,tipo,nulable,llave
0,test_id,integer,False,True
1,correo,string,False,False
2,monto,float,True,False
3,fecha_alta,date,True,False
4,activo,boolean,False,False


## Celda 4 - HU-02: campo requerido faltante -> error claro, sin objeto

Traza -> **HU-02**. La columna `correo` no declara `tipo`. Se espera `ContractSchemaError` indicando **que campo** falta y **en que columna** (indice).

In [4]:
resultado = None
try:
    resultado = load_contract(FIXTURES / "campo_faltante.yaml")
except ContractSchemaError as e:
    print("RECHAZADO con ContractSchemaError (esperado)\n")
    print(e)

print("\nObjeto devuelto:", resultado, "(None = no se construyo objeto a medias)")

RECHAZADO con ContractSchemaError (esperado)

1 validation error for Contract
columnas.1.tipo
  Field required [type=missing, input_value={'nombre': 'correo', 'nul...: False, 'llave': False}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

Objeto devuelto: None (None = no se construyo objeto a medias)


## Celda 5 - HU-03: tipo fuera del enum soportado -> error claro

Traza -> **HU-03**. `tipo: numero_magico` no pertenece al enum. El error debe identificar el **valor invalido** y la **columna afectada**, y enumerar los permitidos.

In [5]:
resultado = None
try:
    resultado = load_contract(FIXTURES / "tipo_invalido.yaml")
except ContractSchemaError as e:
    print("RECHAZADO con ContractSchemaError (esperado)\n")
    print(e)

print("\nTipos soportados:", [t.value for t in TipoDato])
print("Objeto devuelto:", resultado)

RECHAZADO con ContractSchemaError (esperado)

1 validation error for Contract
columnas.0.tipo
  Input should be 'string', 'integer', 'float', 'date' or 'boolean' [type=enum, input_value='numero_magico', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum

Tipos soportados: ['string', 'integer', 'float', 'date', 'boolean']
Objeto devuelto: None


## Celda 6 - HU-04: YAML roto -> error de PARSEO distinguible del de esquema

Traza -> **HU-04**. El fixture tiene indentacion invalida. Debe lanzar `ContractParseError` (no `ContractSchemaError`), demostrando que **por tipo** ambos modos de fallo son distinguibles.

In [6]:
clasificacion = {}
for caso in ["roto.yaml", "campo_faltante.yaml"]:
    try:
        load_contract(FIXTURES / caso)
        clasificacion[caso] = "OK (inesperado)"
    except ContractParseError as e:
        clasificacion[caso] = f"ContractParseError -> {str(e).splitlines()[0]}"
    except ContractSchemaError:
        clasificacion[caso] = "ContractSchemaError (esquema)"

print("Clasificacion por tipo de error (parseo vs esquema):\n")
for caso, etiqueta in clasificacion.items():
    print(f"  {caso:22s} -> {etiqueta}")

assert clasificacion["roto.yaml"].startswith("ContractParseError")
assert clasificacion["campo_faltante.yaml"] == "ContractSchemaError (esquema)"
print("\nOK: parseo y esquema son distinguibles por tipo de excepcion.")

Clasificacion por tipo de error (parseo vs esquema):

  roto.yaml              -> ContractParseError -> YAML sintacticamente invalido: while parsing a block collection
  campo_faltante.yaml    -> ContractSchemaError (esquema)

OK: parseo y esquema son distinguibles por tipo de excepcion.


## Celda 7 - HU-05: columnas duplicadas y lista vacia -> rechazo claro

Traza -> **HU-05**. Dos casos: nombres duplicados y lista vacia. Ambos deben producir `ContractSchemaError` y no devolver objeto.

In [7]:
for caso in ["duplicadas.yaml", "vacio.yaml"]:
    print(f"=== {caso} ===")
    obj = None
    try:
        obj = load_contract(FIXTURES / caso)
    except ContractSchemaError as e:
        # mostramos solo la linea de mensaje relevante para legibilidad
        lineas = [ln for ln in str(e).splitlines() if "duplicadas" in ln or "vacia" in ln]
        print("RECHAZADO (ContractSchemaError):", lineas or str(e).splitlines()[:3])
    print("Objeto devuelto:", obj, "\n")

=== duplicadas.yaml ===
RECHAZADO (ContractSchemaError): ["  Value error, columnas duplicadas por nombre: ['test_id'] [type=value_error, input_value=[{'nombre': 'test_id', 't...: True, 'llave': False}], input_type=list]"]
Objeto devuelto: None 

=== vacio.yaml ===
RECHAZADO (ContractSchemaError): ['  Value error, la lista de columnas no puede estar vacia [type=value_error, input_value=[], input_type=list]']
Objeto devuelto: None 



## Celda 8 - HU-06: frontera con `load_data` (no toca ningun CSV de bronze)

Traza -> **HU-06**. Instrumentamos `open` durante una carga completa y verificamos que **la unica ruta abierta** es el propio `contract_data.yaml`: ninguna ruta bajo `data/bronze/`. El resultado depende solo del YAML.

In [8]:
import builtins

rutas_abiertas = []
_open_original = builtins.open


def _open_espia(archivo, *args, **kwargs):
    rutas_abiertas.append(str(archivo))
    return _open_original(archivo, *args, **kwargs)


builtins.open = _open_espia
try:
    _ = load_contract(FIXTURES / "valido.yaml")
finally:
    builtins.open = _open_original

print("Rutas de archivo abiertas durante load_contract:")
for r in rutas_abiertas:
    print("  -", r)

toca_bronze = any("bronze" in r for r in rutas_abiertas)
solo_contrato = all(r.endswith("valido.yaml") for r in rutas_abiertas)
print("\nToca algun archivo de data/bronze? ->", toca_bronze)
print("Solo abrio el contract_data.yaml?   ->", solo_contrato)
assert not toca_bronze
print("\nOK: la validacion depende unicamente del YAML; frontera con load_data intacta.")

Rutas de archivo abiertas durante load_contract:

Toca algun archivo de data/bronze? -> False
Solo abrio el contract_data.yaml?   -> True

OK: la validacion depende unicamente del YAML; frontera con load_data intacta.


## Celda 9 - HU-07: el core es invocable directamente, sin CLI

Traza -> **HU-07**. `load_contract` se llama como funcion pura de Python (asi la usaran tests, `load_data` y una futura fachada CLI). Mostramos su firma y una invocacion directa.

In [9]:
import inspect

print("Firma del core:", "load_contract", inspect.signature(load_contract))
print("Es callable sin pasar por ninguna CLI:", callable(load_contract))

# invocacion directa (sin argv, sin subprocess, sin fachada)
obj = load_contract(FIXTURES / "valido.yaml")
print("\nInvocacion directa OK -> devuelve", type(obj).__name__, "con", len(obj.columnas), "columnas")
print("Primera columna:", obj.columnas[0].model_dump())

Firma del core: load_contract (path) -> __main__.Contract
Es callable sin pasar por ninguna CLI: True

Invocacion directa OK -> devuelve Contract con 5 columnas
Primera columna: {'nombre': 'test_id', 'tipo': <TipoDato.integer: 'integer'>, 'nulable': False, 'llave': True}


## Celda 10 - HU-08: todos los fixtures son sinteticos (C-01)

Traza -> **HU-08**. Inventario de fixtures: todos viven en el directorio temporal del spike, ninguno bajo `clients/*/data/`, y su contenido son columnas ficticias sin PII.

In [10]:
def es_ruta_de_boveda(ruta: str) -> bool:
    """True si la ruta cae bajo clients/*/data (datos reales, C-01)."""
    r = ruta.replace("\\", "/")
    return ("clients/" in r) or ("/data/" in r) or ("bronze" in r)


inventario = pd.DataFrame(
    [
        {
            "fixture": nombre,
            "origen": "sintetico (escrito en el notebook)",
            "bajo_clients_data": es_ruta_de_boveda(str(FIXTURES / nombre)),
            "proposito": {
                "valido.yaml": "HU-01 contrato bien formado",
                "campo_faltante.yaml": "HU-02 campo requerido faltante",
                "tipo_invalido.yaml": "HU-03 tipo fuera de enum",
                "roto.yaml": "HU-04 sintaxis YAML rota",
                "duplicadas.yaml": "HU-05 nombres duplicados",
                "vacio.yaml": "HU-05 lista vacia",
            }[nombre],
        }
        for nombre in fixtures
    ]
)
print("Todos sinteticos, ninguno bajo clients/*/data:", not inventario["bajo_clients_data"].any())
inventario

Todos sinteticos, ninguno bajo clients/*/data: True


,fixture,origen,bajo_clients_data,proposito
0,valido.yaml,sintetico (escrito en el notebook),False,HU-01 contrato bien formado
1,campo_faltante.yaml,sintetico (escrito en el notebook),False,HU-02 campo requerido faltante
2,tipo_invalido.yaml,sintetico (escrito en el notebook),False,HU-03 tipo fuera de enum
3,roto.yaml,sintetico (escrito en el notebook),False,HU-04 sintaxis YAML rota
4,duplicadas.yaml,sintetico (escrito en el notebook),False,HU-05 nombres duplicados
5,vacio.yaml,sintetico (escrito en el notebook),False,HU-05 lista vacia


## Resumen de cobertura y hallazgos para la spec (paso 6)

| HU | Demostrada en | Resultado visible |
|----|---------------|-------------------|
| HU-01 | Celda 3 | tabla del contrato tipado (nombre/tipo/nulable/llave) |
| HU-02 | Celda 4 | ContractSchemaError: campo `tipo` faltante en columna 1 |
| HU-03 | Celda 5 | ContractSchemaError: `numero_magico` fuera de enum |
| HU-04 | Celda 6 | ContractParseError vs ContractSchemaError distinguibles |
| HU-05 | Celda 7 | rechazo de duplicadas y de lista vacia |
| HU-06 | Celda 8 | espia de `open`: solo abre el YAML, nunca bronze |
| HU-07 | Celda 9 | `load_contract(path) -> Contract` invocable directo |
| HU-08 | Celda 10 | inventario: fixtures 100% sinteticos, sin PII |

### Decisiones tecnicas que quedan para el gate humano / spec
- **Nombres de campos del YAML:** el spike uso `nombre`, `tipo`, `nulable`, `llave` bajo la clave raiz `contract_data.columnas`. La spec debe fijar los nombres definitivos (p. ej. `nulable` vs `nullable`, ES vs EN) y confirmar la clave raiz frente al placeholder de `client_scaffold`.
- **Enum de tipos soportados:** se prototipo `{string, integer, float, date, boolean}` (supuesto de la definition). La lista final se cierra en la spec (HU-03).
- **Duplicados:** se comparo el `nombre` como cadena exacta (sin normalizar mayusculas/espacios), segun el supuesto de la definition. Si se requiere normalizacion, decidir en spec.
- **Distincion parseo vs esquema:** se modelo con dos excepciones (`ContractParseError` / `ContractSchemaError`). La spec debe decidir la jerarquia/normalizacion final de errores y el formato del mensaje accionable.
- **Fachada CLI (HU-07):** el core es suficiente y llamable directo; si se expone `zlk contract validate ...` queda abierto para la spec (puede ser core-only).
- **Riesgo tecnico menor:** los `field_validator` de `columnas` corren tras validar cada `Columna`; por eso un YAML con campo faltante Y duplicados reporta primero el error de columna. Es aceptable, pero la spec puede definir el orden/aggregacion de errores esperado.

> **Siguiente paso: gate humano (paso 5).** El humano revisa y aprueba este spike antes de pasar a `spec_writer` (paso 6). El notebook **no** es fuente de verdad del producto: la spec + TDD reescriben la logica en `app/src/`.